In [1]:
!pip install -q keras-tuner

In [2]:
import tensorflow as tf
import keras_tuner as kt

from tensorflow.keras import layers
from tensorflow.keras import models
from tensorflow.keras import regularizers
from tensorflow.keras import callbacks

import numpy as np
import matplotlib.pyplot as plt

In [3]:
(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

# Normalize
train_images = train_images.astype("float32") / 255.0
test_images = test_images.astype("float32") / 255.0

# Reshape for CNN
train_images = train_images.reshape((-1,28,28,1))
test_images = test_images.reshape((-1,28,28,1))

# One Hot Encoding
train_labels = tf.keras.utils.to_categorical(train_labels)
test_labels = tf.keras.utils.to_categorical(test_labels)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.02),
    layers.RandomZoom(0.05),
    layers.RandomTranslation(0.05,0.05)
])

I0000 00:00:1781120188.998740      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1781120189.004772      58 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:
def build_model(hp):

    model = models.Sequential()

    model.add(layers.Input(shape=(28,28,1)))

    model.add(data_augmentation)

    # ---------------------
    # CNN Block 1
    # ---------------------

    model.add(
        layers.Conv2D(
            filters=hp.Choice(
                "filters_1",
                values=[32,64]
            ),
            kernel_size=(3,3),
            activation='relu',
            padding='same'
        )
    )

    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D())

    model.add(
        layers.Dropout(
            hp.Float(
                "dropout_1",
                min_value=0.1,
                max_value=0.4,
                step=0.1
            )
        )
    )

    # ---------------------
    # CNN Block 2
    # ---------------------

    model.add(
        layers.Conv2D(
            filters=hp.Choice(
                "filters_2",
                values=[64,128]
            ),
            kernel_size=(3,3),
            activation='relu',
            padding='same'
        )
    )

    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D())

    model.add(
        layers.Dropout(
            hp.Float(
                "dropout_2",
                min_value=0.1,
                max_value=0.4,
                step=0.1
            )
        )
    )

    # ---------------------
    # CNN Block 3
    # ---------------------

    model.add(
        layers.Conv2D(
            filters=hp.Choice(
                "filters_3",
                values=[64,128,256]
            ),
            kernel_size=(3,3),
            activation='relu',
            padding='same'
        )
    )

    model.add(layers.BatchNormalization())

    model.add(layers.Flatten())

    # ---------------------
    # Dense Layer
    # ---------------------

    model.add(
        layers.Dense(
            units=hp.Int(
                "dense_units",
                min_value=64,
                max_value=256,
                step=64
            ),
            activation='relu',
            kernel_regularizer=regularizers.l2(
                hp.Choice(
                    "l2",
                    values=[1e-4,1e-3,1e-2]
                )
            )
        )
    )

    model.add(
        layers.Dropout(
            hp.Float(
                "dense_dropout",
                min_value=0.2,
                max_value=0.5,
                step=0.1
            )
        )
    )

    model.add(
        layers.Dense(
            10,
            activation='softmax'
        )
    )

    optimizer = tf.keras.optimizers.Adam(
        learning_rate=hp.Choice(
            "learning_rate",
            values=[1e-2,1e-3,5e-4,1e-4]
        )
    )

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [6]:
tuner = kt.BayesianOptimization(
    hypermodel=build_model,
    objective='val_accuracy',
    max_trials=15,
    num_initial_points=5,
    seed=42,
    overwrite=True,
    directory='cnn_tuner',
    project_name='mnist_cnn'
)

In [7]:
tuner.search_space_summary()

Search space summary
Default search space size: 9
filters_1 (Choice)
{'default': 32, 'conditions': [], 'values': [32, 64], 'ordered': True}
dropout_1 (Float)
{'default': 0.1, 'conditions': [], 'min_value': 0.1, 'max_value': 0.4, 'step': 0.1, 'sampling': 'linear'}
filters_2 (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128], 'ordered': True}
dropout_2 (Float)
{'default': 0.1, 'conditions': [], 'min_value': 0.1, 'max_value': 0.4, 'step': 0.1, 'sampling': 'linear'}
filters_3 (Choice)
{'default': 64, 'conditions': [], 'values': [64, 128, 256], 'ordered': True}
dense_units (Int)
{'default': None, 'conditions': [], 'min_value': 64, 'max_value': 256, 'step': 64, 'sampling': 'linear'}
l2 (Choice)
{'default': 0.0001, 'conditions': [], 'values': [0.0001, 0.001, 0.01], 'ordered': True}
dense_dropout (Float)
{'default': 0.2, 'conditions': [], 'min_value': 0.2, 'max_value': 0.5, 'step': 0.1, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01

In [8]:
early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7
)

In [ ]:
tuner.search(
    train_images,
    train_labels,
    validation_split=0.15,
    epochs=15,
    batch_size=128,
    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

Trial 4 Complete [00h 01m 59s]
val_accuracy: 0.9933333396911621

Best val_accuracy So Far: 0.9941111207008362
Total elapsed time: 00h 08m 07s

Search: Running Trial #5

Value             |Best Value So Far |Hyperparameter
64                |32                |filters_1
0.1               |0.2               |dropout_1
64                |128               |filters_2
0.3               |0.2               |dropout_2
64                |128               |filters_3
128               |128               |dense_units
0.01              |0.0001            |l2
0.2               |0.2               |dense_dropout
0.0001            |0.0005            |learning_rate

Epoch 1/15


E0000 00:00:1781120698.158126      58 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1/dropout_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


399/399 ━━━━━━━━━━━━━━━━━━━━ 12s 20ms/step - accuracy: 0.8234 - loss: 2.6266 - val_accuracy: 0.8954 - val_loss: 2.2231 - learning_rate: 1.0000e-04
Epoch 2/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.9487 - loss: 1.5025 - val_accuracy: 0.9826 - val_loss: 1.0916 - learning_rate: 1.0000e-04
Epoch 3/15
399/399 ━━━━━━━━━━━━━━━━━━━━ 7s 19ms/step - accuracy: 0.9657 - loss: 0.9145 - val_accuracy: 0.9864 - val_loss: 0.6518 - learning_rate: 1.0000e-04
Epoch 4/15
355/399 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.9718 - loss: 0.6315

In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]

print("Best Parameters")

for param in best_hp.values:
    print(param, ":", best_hp.values[param])

In [ ]:
best_model = tuner.get_best_models(1)[0]

In [ ]:
history = best_model.fit(
    train_images,
    train_labels,
    validation_split=0.15,
    epochs=30,
    batch_size=128,
    callbacks=[
        early_stopping,
        reduce_lr
    ]
)

In [ ]:
test_loss, test_acc = best_model.evaluate(
    test_images,
    test_labels
)

print("Test Accuracy:", test_acc)